In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
dbutils.widgets.text("batch_id","1","BATCH ID")
source_schema="landing"
target_schema="bronze"
landing_base_path=f"/Volumes/{catalog_name}/{source_schema}/PWG/"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:

CUSTOMER_DOMAIN_FILES=[
    {
        "batch":["1"],
        "type":"xml",
        "landing_name":"customermgmt"
    },
    {     
        "batch":["2","3"],
        "type":"csv_pipe",
        "landing_name":"customer"
    },
    {     
        "batch":["1","2","3"],
        "type":"json",
        "landing_name":"prospect"
    }
    ,
    {
        "batch":["1","2","3"],
        "type":"csv_pipe",
        "landing_name":"watchhistory"
    }
]

In [0]:
def landing_to_bronze(batch_id):
    try:
        print(f"Landing to Bronze For Batch {batch_id}")
        for f in CUSTOMER_DOMAIN_FILES:
            if batch_id in f["batch"]:
                df=spark.read.parquet(f"{landing_base_path}/Batch{batch_id}/{f["landing_name"]}")
                df= df.select([
                    col(c).cast(StringType()).alias(c)
                    for c in df.columns
                ])
                df=df.drop("_landing_ts")
                df=df.withColumn("_ingest_ts",current_timestamp())
                run_id=df.select("_run_id").first()[0]
                # print(run_id)
                # df.limit(10).display()
                source_count=df.count()
                print(f"writing table {f["landing_name"]} to bronze layer ")
                df.write.mode("append")\
                    .option("mergeSchema","true")\
                    .partitionBy("_batch")\
                    .saveAsTable(f"{catalog_name}.{target_schema}.{f["landing_name"]}")
                target_count=spark.read.table(f"{catalog_name}.{target_schema}.{f["landing_name"]}").count()     
                print(f"{f['landing_name']} written successfully to bronze layer with {target_count} rows")
                log_pipeline_recon(
                    spark=spark,
                    run_id=run_id,
                    batch_id=batch_id,
                    domain="CUSTOMER",
                    table_name=f['landing_name'],
                    source_layer="landing",
                    target_layer="bronze",
                    source_count=source_count,
                    target_count=target_count
                )
                log_audit_event(
                    spark=spark,
                    run_id=run_id,
                    batch=batch_id,
                    layer="bronze",
                    table_name=f['landing_name'],
                    operation="APPEND",
                    rows_affected=target_count
                )
    except Exception as e:
        raise e

In [0]:
landing_to_bronze(batch_id)